# İleri YZ – WarmUp Ödevi: MemGraphRAG Uygulamasının Çalıştırılması

**Öğrenci:** Yusuf Utar
**Bölüm:** Ankara Üniversitesi, Yapay Zeka ve Veri Mühendisliği
**Kaynak kod:** [XMUDeepLIT/MemGraphRAG](https://github.com/XMUDeepLIT/MemGraphRAG) (KDD'26, [arXiv:2606.00610](https://arxiv.org/abs/2606.00610))

| Bileşen | Bu notebook'ta kullanılan |
|---|---|
| **Dil modeli (LLM)** | **Qwen2.5-7B-Instruct** (`qwen2.5:7b`, 4-bit GGUF) — Colab GPU'sunda **Ollama** ile yerel olarak, OpenAI-uyumlu uç nokta üzerinden |
| Embedding modeli | `BAAI/bge-large-en-v1.5` (repo README'sindeki model) |
| Veri seti / örnek | Repo ile gelen **2WikiMultiHopQA**'dan türetilen küçük alt küme (çok-adımlı sorular) |
| Çalıştırılan repo adımları | `code/index.py` (indeksleme) → `code/retrieval_dataset_test.py` (retrieval + soru-cevap) → repo'nun `evaluation` modülü (EM / F1) |
| Ortam | Google Colab, T4 GPU |

**İçindekiler**

1. MemGraphRAG'ın çalışma mantığı
2. Ayarlar ve ortam kontrolü
3. Projenin Colab'a aktarılması
4. Bağımlılıkların kurulumu
5. Dil modelinin hazırlanması ve sisteme entegrasyonu
6. Embedding modelinin hazırlanması
7. Veri setinin hazırlanması
8. İndeksleme: üç katmanlı belleğin ve grafın kurulması
9. Üretilen bellek ve grafın incelenmesi
10. Retrieval + soru-cevap örneğinin çalıştırılması
11. Sonuçlar ve değerlendirme
12. Klasik RAG ile küçük bir karşılaştırma
13. Kullanılan dil modeli ve tercih gerekçesi
14. Karşılaşılan hatalar ve çözümleri
15. Teknik değerlendirme: MemGraphRAG, klasik RAG ve GraphRAG'dan nasıl ayrışıyor?

> **Çalıştırma notu:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçildikten sonra *Tümünü çalıştır* yeterlidir. API anahtarı gerekmez.

## 1. MemGraphRAG'ın çalışma mantığı

Aşağıdaki özet, makalenin yanı sıra doğrudan repodaki kodun (`code/src/MemGraphRAG.py`, `code/src/Memory.py`, `code/src/prompts/prompt.py`) okunmasıyla çıkarılmıştır.

<p align="center"><img src="https://raw.githubusercontent.com/XMUDeepLIT/MemGraphRAG/main/framework.png" width="85%"></p>

**Çözmeye çalıştığı problem.** Klasik GraphRAG yöntemleri her metin parçasından (chunk) bilgiyi *birbirinden bağımsız* çıkarır. Bütüne bakan bir bağlam olmadığı için ortaya çıkan graf tematik olarak tutarsız, mantıksal olarak çelişkili ve yapısal olarak parçalı olabilir; bu da retrieval kalitesini düşürür. MemGraphRAG, çıkarım sürecine **ortak bir bellek** ekleyerek bu sorunu hedefler.

**Üç katmanlı bellek** (`ThreeLayerMemory`):

| Katman | İçerik | Örnek |
|---|---|---|
| **Schema (şema) katmanı** | Soyut ontoloji üçlüleri: `(baş tipi, ilişki, kuyruk tipi)` | `(<PERSON>, married to, <PERSON>)` |
| **Fact (olgu) katmanı** | Metinden çıkarılan somut üçlüler | `(lothair ii, married to, teutberga)` |
| **Passage (pasaj) katmanı** | Olguları destekleyen özgün metin parçaları | 256 token'lık chunk'lar |

Katmanlar çift yönlü indekslerle bağlıdır: `Schema —1:N→ Fact —N:M→ Passage`.

**İndeksleme hattı** (`MemGraphRAG.index_with_memory`), sırasıyla:

1. **OpenIE** – Her chunk için LLM önce varlıkları (NER), sonra bu varlıklara koşullu üçlüleri çıkarır.
2. **Şema çıkarımı** (*ontology induction*) – Her olgu, kaynak pasajı bağlam alınarak LLM'e verilir ve tip düzeyinde bir şema üçlüsüne soyutlanır.
3. **Ontoloji filtreleme** – En düşük frekanslı şemalar (varsayılan: alt %20) ve onlara bağlı olgular gürültü kabul edilip atılır.
4. **Çelişki tespiti** – Aynı *(özne, ilişki)* çiftine sahip ama nesnesi farklı olgular ile ters ilişkiler aday olarak eşlenir; LLM, pasaj kanıtlarına bakarak bunların *gerçek (hard)* çelişki olup olmadığına karar verir (mutual / temporal / granularity).
5. **Çelişki çözümü** – Çelişen olgular bağlı bileşenlere ayrılır; LLM her bileşen için olguları `kept / modified / discarded` olarak işaretler ve bellek yeniden kurulur.
6. **Bellekten graf türetme** – Çelişkileri çözülmüş bellekten **tip**, **varlık** ve **pasaj** düğümlerinden oluşan graf kurulur (varlık–varlık, pasaj–varlık, varlık–tip, tip–tip ve embedding benzerliğine dayalı eş anlamlılık kenarları).

**Retrieval ve cevaplama** (`retrieve` → `qa`):

1. Soru embedding'i ile tüm **olgu** embedding'leri arasındaki benzerlik hesaplanır; eşiği geçen olgular seçilir.
2. Seçilen olguların varlıkları grafta *tohum* düğüm olur; ayrıca her pasaj düğümüne dense benzerliği oranında küçük bir ağırlık verilir.
3. Bu ağırlıklarla graf üzerinde **Personalized PageRank (PPR)** çalıştırılır; en yüksek skorlu pasajlar döndürülür.
4. Pasajlar tek örnekli (one-shot) bir düşünce-zinciri istemiyle LLM'e verilir, `Answer:` sonrasındaki kısım cevap olarak alınır.

Repo, HippoRAG kod tabanının üzerine kuruludur; OpenIE, PPR tabanlı retrieval ve QA istemleri oradan gelir. MemGraphRAG'ın eklediği kısım 2–6. adımlardaki bellek hattıdır.

## 2. Ayarlar ve ortam kontrolü

Tüm ayarlar tek hücrede toplandı. Dil modelini değiştirmek için yalnızca `LLM_NAME` / `LLM_BASE_URL` değerlerini değiştirmek yeterlidir (ör. OpenAI için `LLM_NAME="gpt-4o-mini"`, `LLM_BASE_URL=None` ve `OPENAI_API_KEY`).

In [ ]:
import os, sys, json, time, re, subprocess, textwrap

# ---- Dil modeli (LLM) ----
LLM_NAME      = "qwen2.5:7b"                  # Ollama model etiketi (Qwen2.5-7B-Instruct, Q4_K_M)
LLM_BASE_URL  = "http://localhost:11434/v1"   # Ollama'nın OpenAI-uyumlu uç noktası
USE_OLLAMA    = True                          # False yapılırsa 5. bölümdeki Ollama kurulumu atlanır (harici API kullanımı)

# ---- Embedding modeli ----
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

# ---- Repo ----
REPO_URL    = "https://github.com/XMUDeepLIT/MemGraphRAG.git"
REPO_COMMIT = "cd6fabda1ec31a302bb283089e17c087ab7713e8"   # tekrarlanabilirlik için sabitlenen commit
REPO_DIR    = os.path.abspath("MemGraphRAG")

# ---- Mini veri seti (2WikiMultiHopQA alt kümesi) ----
N_QUESTIONS   = 10   # kullanılacak soru sayısı
N_DISTRACTORS = 2    # her soru için eklenecek çeldirici paragraf sayısı
DATA_NAME     = "mini2wiki"

# ---- Çalıştırma parametreleri ----
CHUNK_SIZE, CHUNK_OVERLAP = 256, 32   # README'deki değerler
MEMORY_MAX_WORKERS = 4                # tek GPU'daki yerel model için (repo varsayılanı 20)
RETRIEVAL_TOP_K, QA_TOP_K = 10, 3     # mini derlem küçük olduğu için LLM'e yalnızca ilk 3 pasaj verilir
FACT_SIM_THRESHOLD = 0.4              # README'deki değer

CORPUS_PATH    = os.path.join(REPO_DIR, "dataset", DATA_NAME, f"{DATA_NAME}.txt")
QUESTIONS_PATH = os.path.join(REPO_DIR, "dataset", DATA_NAME, f"{DATA_NAME}_questions.json")
PARAS_PATH     = os.path.join(REPO_DIR, "dataset", DATA_NAME, f"{DATA_NAME}_paragraphs.json")
SAVE_DIR       = os.path.join(REPO_DIR, "outputs", DATA_NAME)
RESULT_PATH    = os.path.join(REPO_DIR, "results", DATA_NAME, "qa_results.json")

# Ollama kimlik doğrulaması istemez; OpenAI istemcisi boş anahtarı kabul etmediği için yer tutucu veriyoruz.
if USE_OLLAMA:
    os.environ["OPENAI_API_KEY"] = "ollama"
print("LLM:", LLM_NAME, "| uç nokta:", LLM_BASE_URL, "| embedding:", EMBEDDING_MODEL)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!python --version
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU bulunamadı: Çalışma zamanı türünü T4 GPU olarak değiştirin."

## 3. Projenin Colab'a aktarılması

Repo klonlanır ve tekrarlanabilirlik için bu çalışmanın yapıldığı commit'e sabitlenir.

In [ ]:
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout -q {REPO_COMMIT}
!git -C {REPO_DIR} log -1 --format="commit %h | %ad | %s" --date=short
print()
!ls {REPO_DIR} {REPO_DIR}/code {REPO_DIR}/dataset

## 4. Bağımlılıkların kurulumu

Repo'nun `requirements.txt` dosyası şu sürümleri sabitliyor: `torch==2.5.1`, `transformers==4.45.2`, `vllm==0.6.6.post1`, `gritlm==1.0.2`, `litellm`, `boto3` …

Bu dosyayı Colab'da olduğu gibi kurmak yerine **yalnızca gerçekten gereken paketleri** kurdum, çünkü:

- `vllm` yalnızca *offline OpenIE* modu için gerekli (README: *"For offline OpenIE, also install vllm"*). Biz OpenAI-uyumlu uç nokta kullanan *online* modu çalıştırıyoruz. `vllm==0.6.6.post1` ayrıca Colab'daki hazır `torch`'u eski sürüme çekip çalışma zamanını bozuyor.
- `torch` ve `transformers` Colab'da zaten kurulu; sabitlenen eski sürümlere indirmek gerekmiyor.
- `litellm` ve `boto3` kodda hiçbir yerde import edilmiyor.
- `gritlm` kullanılmasa da `code/src/embedding_model/__init__.py` içinde koşulsuz import edildiği için **gerekli**; fakat kendi bağımlılıkları (`wandb`, `mteb`, `datasets`) gereksiz olduğundan `--no-deps` ile kurdum.

In [ ]:
!pip -q install openai python-igraph tenacity tiktoken filelock httpx pyarrow accelerate
!pip -q install --no-deps gritlm==1.0.2

import importlib.metadata as md_
for pkg in ["openai", "igraph", "tenacity", "tiktoken", "gritlm", "transformers", "torch", "accelerate"]:
    print(f"{pkg:13s}", md_.version(pkg))

Kurulumun repo tarafından kabul edildiğini, indeksleme betiğinin yardım çıktısıyla doğruluyorum (bu komut `src` paketindeki tüm modülleri import eder):

In [ ]:
!cd {REPO_DIR} && python code/index.py --help | head -n 12

### Küçük dayanıklılık yaması

Bellek hattındaki üç LLM adımı (şema çıkarımı, çelişki tespiti, çelişki çözümü) cevabı `_memory_json` ile JSON olarak ayrıştırıyor. Bu fonksiyon tek bir bozuk JSON'da istisna fırlatıyor ve yüzlerce çağrılık indeksleme **tümüyle** duruyor. Küçük açık kaynak modellerde bu risk daha yüksek olduğu için fonksiyonu, ayrıştırılamayan cevabı *uyarı verip atlayacak* şekilde sarmaladım (ilgili olgu şemasız kalır ve ontoloji filtresinde elenir). Yamanın kaç kez devreye girdiği indeksleme sonunda raporlanıyor.

In [ ]:
target = os.path.join(REPO_DIR, "code", "src", "MemGraphRAG.py")
src = open(target, encoding="utf-8").read()
OLD = "    @staticmethod\n    def _memory_json(response: str) -> Dict[str, Any]:\n"
NEW = (
    "    @staticmethod\n"
    "    def _memory_json(response: str) -> Dict[str, Any]:\n"
    "        # [colab-patch] ayrıştırılamayan LLM cevabı tüm indekslemeyi düşürmesin\n"
    "        try:\n"
    "            parsed = MemGraphRAG._memory_json_strict(response)\n"
    "            return parsed if isinstance(parsed, dict) else {}\n"
    "        except Exception:\n"
    "            logger.warning(\"[colab-patch] unparseable LLM JSON skipped: %r\", str(response)[:160])\n"
    "            return {}\n"
    "\n"
    "    @staticmethod\n"
    "    def _memory_json_strict(response: str) -> Dict[str, Any]:\n"
)
if "_memory_json_strict" in src:
    print("Yama zaten uygulanmış.")
else:
    assert src.count(OLD) == 1, "Beklenen fonksiyon imzası bulunamadı (repo sürümü değişmiş olabilir)."
    open(target, "w", encoding="utf-8").write(src.replace(OLD, NEW))
    print("Yama uygulandı:", os.path.relpath(target, REPO_DIR))
!git -C {REPO_DIR} --no-pager diff --stat --no-color -- code/src/MemGraphRAG.py

## 5. Dil modelinin hazırlanması ve sisteme entegrasyonu

**Seçilen model: Qwen2.5-7B-Instruct** (gerekçe 13. bölümde).

Repo'nun LLM katmanı (`code/src/llm/openai_gpt.py`) OpenAI istemcisini kullanıyor ve `--llm-base-url` ile herhangi bir **OpenAI-uyumlu** uç noktaya yönlendirilebiliyor. Bu yüzden modeli Colab GPU'sunda **Ollama** ile sunup `http://localhost:11434/v1` adresini vermek, repo kodunda hiçbir değişiklik yapmadan entegrasyon için yeterli:

- Model adında `gpt` geçmediğinde kod `max_completion_tokens` yerine `max_tokens` gönderiyor → Ollama ile uyumlu.
- `base_url` içinde `localhost` varsa ve anahtar tanımlı değilse kod `OPENAI_API_KEY`'i kendisi yer tutucuyla dolduruyor.

Ollama sunucu ayarları:

- `OLLAMA_CONTEXT_LENGTH=8192` — varsayılan bağlam penceresi QA istemi (tek örnek + 3 pasaj) ve kanıtlı çelişki istemleri için dar kalabildiğinden genişletildi; aksi halde istem sessizce kırpılıyor.
- `OLLAMA_NUM_PARALLEL=4` — repo çağrıları iş parçacıklarıyla paralel gönderiyor.

In [ ]:
import requests

def ollama_alive():
    try:
        return requests.get("http://127.0.0.1:11434/api/version", timeout=2).ok
    except Exception:
        return False

if USE_OLLAMA:
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
        !sudo apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
        !curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 4
    if not ollama_alive():
        env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_NUM_PARALLEL="4",
                   OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="-1")
        ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                                       stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_alive():
                break
            time.sleep(1)
    assert ollama_alive(), "Ollama sunucusu başlatılamadı (ollama.log dosyasına bakın)."
    print("Ollama sürümü:", requests.get("http://127.0.0.1:11434/api/version").json()["version"])
    t0 = time.time()
    !ollama pull {LLM_NAME} 2>&1 | tail -n 1
    print(f"Model indirme süresi: {time.time() - t0:.0f} sn")
    !ollama list

**Entegrasyon testi.** Repo'nun kullandığı arayüzün aynısıyla (OpenAI `chat.completions`) modele, OpenIE adımındakine benzer bir JSON çıkarım istemi gönderiyorum.

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=LLM_BASE_URL, api_key=os.environ.get("OPENAI_API_KEY", "ollama"))

t0 = time.time()
resp = client.chat.completions.create(
    model=LLM_NAME, temperature=0, max_tokens=200,
    messages=[
        {"role": "system", "content": "Your task is to extract named entities from the given paragraph. Respond with a JSON list of entities."},
        {"role": "user", "content": "Lothair II was the king of Lotharingia from 855. He was married to Teutberga, daughter of Boso the Elder."},
    ],
)
dt = time.time() - t0
print(resp.choices[0].message.content)
print(f"\nmodel={resp.model} | prompt={resp.usage.prompt_tokens} tok | completion={resp.usage.completion_tokens} tok | {dt:.1f} sn (ilk çağrı model yüklemesini içerir)")

In [ ]:
!ollama ps
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6. Embedding modelinin hazırlanması

README'de önerilen `bge-large-en-v1.5` kullanıldı. Repo, embedding sınıfını model adındaki alt dizgeye göre seçiyor (`_get_embedding_model_class`); adın `bge-large-en-v1.5` içermesi gerekiyor. Model Hugging Face'ten önceden indirilip kısa bir anlamsal benzerlik testiyle doğrulanıyor.

In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download(EMBEDDING_MODEL, allow_patterns=["*.json", "*.txt", "model.safetensors"])
print("İndirilen model:", path)
!du -sh {path}/

In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, "code"))
import numpy as np
from src.embedding_model.BGE import BGELargeEnV15EmbeddingModel
from src.utils.config_utils import BaseConfig

# Repo'nun kendi embedding sınıfı (HF AutoModel + mean pooling + L2 normalizasyon)
embedder = BGELargeEnV15EmbeddingModel(
    global_config=BaseConfig(embedding_model_name=EMBEDDING_MODEL, embedding_batch_size=8, save_dir=SAVE_DIR),
    embedding_model_name=EMBEDDING_MODEL,
)
sents = ["Who was the mother of Lothair II?",
         "Lothair II was the second son of Emperor Lothair I and Ermengarde of Tours.",
         "Aas Ka Panchhi is a 1961 Hindi movie produced by J. Om Prakash."]
E = embedder.batch_encode(sents, norm=True)
print("Embedding boyutu:", E.shape, "| cihaz:", embedder.embedding_model.device)
print("soru ↔ ilgili cümle   :", round(float(E[0] @ E[1]), 3))
print("soru ↔ ilgisiz cümle  :", round(float(E[0] @ E[2]), 3))

## 7. Veri setinin hazırlanması

Repo dört veri setiyle geliyor (`dataset/` klasörü): **HotpotQA**, **MuSiQue**, **2WikiMultiHopQA** ve **medical**. Her biri için düz metin derlem (`*.txt`) ve soru dosyası (`*.json`) var.

In [ ]:
for root, _, files in sorted(os.walk(os.path.join(REPO_DIR, "dataset"))):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.relpath(p, REPO_DIR):55s} {os.path.getsize(p) / 1e6:6.1f} MB")

**Neden alt küme?** MemGraphRAG indekslemesi LLM açısından yoğun: her chunk için 2 çağrı (NER + üçlü çıkarımı), ardından **her olgu için 1** şema çağrısı ve çelişki adayları için ek çağrılar yapılıyor. 2Wiki derleminin tamamı (~6 bin paragraf, on binlerce olgu) Colab T4 üzerindeki 7B'lik yerel bir modelle saatler sürerdi. Bu yüzden repo'nun **2WikiMultiHopQA** verisinden, boru hattının *tüm* aşamalarını uçtan uca çalıştıracak küçük bir alt küme türettim:

- İlk `N_QUESTIONS` soru alınır.
- Her sorunun **destekleyici paragrafları** (cevap için gereken 2–4 paragraf) ile birlikte `N_DISTRACTORS` adet **çeldirici** paragraf derleme eklenir.
- Derlem, repo'daki `.txt` dosyalarıyla aynı biçimde (`Başlık\nMetin\n\n`) yazılır.
- Sorular, README'deki *"Dataset Format"* bölümünde tarif edilen, türe göre gruplanmış biçime dönüştürülür (`retrieval_dataset_test.py` 2Wiki'nin ham biçimini değil bu biçimi bekliyor).

2Wiki'yi seçme nedenim soruların **çok-adımlı** (multi-hop) olması: cevap için birden fazla paragrafın birleştirilmesi gerekiyor; grafın katkısı en çok bu tür sorularda görülür.

In [ ]:
src_path = os.path.join(REPO_DIR, "dataset", "2wikimutlhopqa", "2wikimultihopqa.json")
data = json.load(open(src_path, encoding="utf-8"))
print("2WikiMultiHopQA soru sayısı:", len(data), "| alanlar:", list(data[0].keys()))

corpus, paragraphs, grouped = [], {}, {}
for item in data[:N_QUESTIONS]:
    sup_titles = list(dict.fromkeys(t for t, _ in item["supporting_facts"]))
    ctx = {title: " ".join(s.strip() for s in sents) for title, sents in item["context"]}
    distractors = [t for t in ctx if t not in sup_titles][:N_DISTRACTORS]
    for title in sup_titles + distractors:
        if title in ctx and title not in paragraphs:
            paragraphs[title] = ctx[title]
            corpus.append(f"{title}\n{ctx[title]}")
    grouped.setdefault(item["type"], []).append(
        {"id": item["_id"], "question": item["question"], "answer": item["answer"], "evidence": sup_titles})

os.makedirs(os.path.dirname(CORPUS_PATH), exist_ok=True)
open(CORPUS_PATH, "w", encoding="utf-8").write("\n\n".join(corpus) + "\n")
json.dump([{"source": "2wikimultihopqa-mini", "questions": grouped}],
          open(QUESTIONS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(paragraphs, open(PARAS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"Mini derlem: {len(corpus)} paragraf, {sum(len(c.split()) for c in corpus)} kelime → {os.path.relpath(CORPUS_PATH, REPO_DIR)}")
print("Soru türleri:", {k: len(v) for k, v in grouped.items()})
print("\n--- Derlemin başı ---\n" + open(CORPUS_PATH, encoding="utf-8").read()[:420] + " ...")
print("\n--- Örnek soru ---\n" + json.dumps(next(iter(grouped.values()))[0], ensure_ascii=False, indent=2))

## 8. İndeksleme: üç katmanlı belleğin ve grafın kurulması

README'deki **"3. Run Indexing"** komutu, yalnızca yollar ve model adları değiştirilerek çalıştırılıyor. README'ye göre farklar:

- `--artifact-mode detailed`: her ara aşamanın çıktısı (ilk bellek, filtrelenmiş bellek, çelişki tespiti/çözümü) diske yazılsın diye — 9. bölümde bunları inceliyoruz.
- `--memory-max-workers 4`: varsayılan 20 paralel işçi, tek GPU'daki yerel model için gereksiz kuyruk oluşturuyor.

`run_cli` yardımcı fonksiyonu betiği olduğu gibi çalıştırır; yalnızca çıktıyı okunur kılmak için her HTTP isteğinde basılan `httpx` satırlarını gizler ve ilerleme çubuklarını seyrek basar. Tam çıktı log dosyasına yazılır.

In [ ]:
def run_cli(args, log_path, progress_every=30):
    # Repo betiğini çalıştırır; httpx satırlarını gizler, tqdm çubuklarını seyrek basar, tam çıktıyı log'a yazar.
    env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    print("$ python " + " ".join(args) + "\n")
    t0, last, done = time.time(), {}, set()
    proc = subprocess.Popen([sys.executable, *args], cwd=REPO_DIR, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    with open(log_path, "w", encoding="utf-8") as log:
        for line in proc.stdout:            # text kipinde '\r' de satır sonu sayılır (tqdm)
            log.write(line)
            line = line.strip()
            if not line or "httpx" in line:
                continue
            m = re.match(r"^([^:|]{1,40}):\s+(\d+)%\|", line)
            if m:
                key, pct = m.group(1), m.group(2)
                if pct == "100":
                    if key in done:
                        continue
                    done.add(key)
                elif time.time() - last.get(key, 0) < progress_every:
                    continue
                last[key] = time.time()
                line = re.sub(r"\|[^|]*\|", " |", line, count=1)
            print(line[:260])
    rc = proc.wait()
    print(f"\n[çıkış kodu {rc} | süre {time.time() - t0:.0f} sn | tam log: {log_path}]")
    assert rc == 0, f"Komut hata ile bitti, ayrıntı: {log_path}"
    return time.time() - t0

index_args = [
    "code/index.py",
    "--corpus", CORPUS_PATH,
    "--save-dir", SAVE_DIR,
    "--llm-name", LLM_NAME,
    "--llm-base-url", LLM_BASE_URL,
    "--embedding-model", EMBEDDING_MODEL,
    "--tokenizer", EMBEDDING_MODEL,
    "--chunk-size", str(CHUNK_SIZE),
    "--chunk-overlap", str(CHUNK_OVERLAP),
    "--artifact-mode", "detailed",
    "--memory-max-workers", str(MEMORY_MAX_WORKERS),
]
index_seconds = run_cli(index_args, "index.log")

In [ ]:
log_text = open("index.log", encoding="utf-8").read()
print("Ayrıştırılamayıp atlanan LLM cevabı (yama) :", log_text.count("[colab-patch]"))
print("OpenIE aşamasında hata veren chunk          :", log_text.count("Exception for chunk"))
print("\nÜretilen dosyalar (README'deki çıktı ağacıyla aynı yapı):")
for root, dirs, files in sorted(os.walk(SAVE_DIR)):
    depth = os.path.relpath(root, SAVE_DIR).count(os.sep) + (0 if root == SAVE_DIR else 1)
    if depth > 2 or "llm_cache" in root:
        continue
    print("  " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        print("  " * (depth + 1) + f"{f}  ({os.path.getsize(os.path.join(root, f)) / 1e3:.0f} KB)")

## 9. Üretilen bellek ve grafın incelenmesi

### 9.1 Belleğin aşamalar boyunca değişimi

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 110)

def load(name):
    return json.load(open(os.path.join(SAVE_DIR, name), encoding="utf-8"))

stages = [("1) OpenIE sonrası ilk bellek", "initial_memory.json"),
          ("2) Şema çıkarımı sonrası", "initial_memory_with_schema.json"),
          ("3) Ontoloji filtresi sonrası", "memory_after_ontology_filtering.json"),
          ("4) Çelişki çözümü sonrası (nihai)", "memory.json")]
rows = []
for label, fname in stages:
    m = load(fname)
    rows.append({"Aşama": label, "Şema": len(m["schema_layer"]), "Olgu": len(m["fact_layer"]),
                 "Pasaj": len(m["passage_layer"])})
display(pd.DataFrame(rows).set_index("Aşama"))

memory = load("memory.json")
schemas, facts, passages = memory["schema_layer"], memory["fact_layer"], memory["passage_layer"]

### 9.2 Şema katmanı: en sık görülen ontoloji üçlüleri ve bağlı olgular

In [ ]:
top = sorted(schemas, key=lambda s: -s["frequency"])[:12]
display(pd.DataFrame([{"şema (baş tipi, ilişki, kuyruk tipi)": " | ".join(s["content"]),
                       "frekans": s["frequency"],
                       "örnek olgu": " | ".join(facts[s["fact_indices"][0]]["content"]) if s["fact_indices"] else ""}
                      for s in top]))

### 9.3 Katmanlar arası bağlantı: bir pasaj → olguları → şemaları

In [ ]:
p = max(passages, key=lambda x: len(x["fact_indices"]))
print("PASAJ", p["idx"], "(", p["chunk_id"], ")\n" + textwrap.fill(p["content"][:600] + " ...", 120), "\n")
display(pd.DataFrame([{"olgu (fact)": " | ".join(facts[i]["content"]),
                       "şema (schema)": " | ".join(schemas[facts[i]["schema_idx"]]["content"]) if facts[i]["schema_idx"] >= 0 else "-",
                       "geçtiği pasajlar": facts[i]["passage_indices"]}
                      for i in p["fact_indices"][:12]]))

### 9.4 Çelişki tespiti ve çözümü

In [ ]:
det, res = load("conflict_detection_results.json"), load("conflict_resolution_results.json")
print("Çelişki tespiti özeti:", json.dumps(det["summary"], ensure_ascii=False))
print("Çelişki çözümü özeti :", json.dumps(res["summary"], ensure_ascii=False))

if det["conflict_items"]:
    display(pd.DataFrame([{"hedef olgu": " | ".join(map(str, c["target_triple"])),
                           "çelişen olgu": " | ".join(map(str, c["other_triple"])),
                           "tür": c["conflict_type"], "gerekçe": str(c["reason"])[:160]}
                          for c in det["conflict_items"][:15]]))
    display(pd.DataFrame([{"olgu id": a["fact_id"], "karar": a["action"],
                           "yeni üçlü": a["resolved_triple"], "gerekçe": str(a["reason"])[:160]}
                          for a in res["applied_actions"][:15]]))
else:
    print("\nBu mini derlemde model, aday çiftlerin hiçbirini 'hard conflict' olarak işaretlemedi "
          f"({det['summary']['num_detection_tasks']} aday incelendi).")

### 9.5 Bellekten türetilen graf

In [ ]:
graph = json.load(open(os.path.join(SAVE_DIR, "graph_from_memory", "memory_graph.json"), encoding="utf-8"))
display(pd.DataFrame([graph["stats"]]).T.rename(columns={0: "adet"}))

In [ ]:
%matplotlib inline
import networkx as nx
import matplotlib.pyplot as plt

nodes = {n["id"]: n for n in graph["nodes"]}
G = nx.Graph()
for e in graph["edges"]:
    if e["type"] != "entity_similarity":          # okunabilirlik için eş anlamlılık kenarları çizilmiyor
        G.add_edge(e["source"], e["target"], type=e["type"])

# En çok ilişkiye sahip varlığın 2 adımlık komşuluğu
ent_deg = {n: sum(1 for _, _, d in G.edges(n, data=True) if d["type"] == "entity_relation")
           for n in G if nodes[n]["layer"] == "entity"}
center = max(ent_deg, key=ent_deg.get)
ego = nx.ego_graph(G, center, radius=2)
keep = [center] + sorted((n for n in ego if n != center), key=lambda n: nx.shortest_path_length(ego, center, n))[:45]
H = ego.subgraph(keep)

colors = {"type": "#d95f02", "entity": "#1b9e77", "passage": "#7570b3"}
label = lambda n: (nodes[n]["content"][:22] + "…") if nodes[n]["layer"] == "passage" else nodes[n]["content"][:28]
pos = nx.spring_layout(H, seed=7, k=0.9)
plt.figure(figsize=(14, 9))
nx.draw_networkx_edges(H, pos, alpha=0.35)
for layer, col in colors.items():
    ns = [n for n in H if nodes[n]["layer"] == layer]
    nx.draw_networkx_nodes(H, pos, nodelist=ns, node_color=col, node_size=420 if layer != "entity" else 220, label=layer)
nx.draw_networkx_labels(H, pos, labels={n: label(n) for n in H}, font_size=8)
plt.legend(title="düğüm katmanı", loc="upper left"); plt.axis("off")
plt.title(f"Bellekten türetilen grafın bir kesiti — merkez varlık: '{nodes[center]['content']}'")
plt.show()

## 10. Retrieval + soru-cevap örneğinin çalıştırılması

README'deki **"4. Run Retrieval and Question Answering"** komutu çalıştırılıyor. `--skip-fact-rerank true`, `--fact-similarity-threshold 0.4` ve `--use-raw-threshold-filter true` README'deki değerlerdir (olgular LLM ile yeniden sıralanmaz; ham benzerlik eşiğiyle süzülür). README'den tek fark `--qa-top-k 3`: mini derlemde yalnızca ~20 chunk olduğu için varsayılan 10 pasajı LLM'e vermek retrieval'ı anlamsızlaştırırdı.

In [ ]:
qa_args = [
    "code/retrieval_dataset_test.py",
    "--questions", QUESTIONS_PATH,
    "--save-dir", SAVE_DIR,
    "--output", RESULT_PATH,
    "--llm-name", LLM_NAME,
    "--llm-base-url", LLM_BASE_URL,
    "--embedding-model", EMBEDDING_MODEL,
    "--question-type", "all",
    "--sample-num", "0",
    "--skip-fact-rerank", "true",
    "--fact-similarity-threshold", str(FACT_SIM_THRESHOLD),
    "--use-raw-threshold-filter", "true",
    "--retrieval-top-k", str(RETRIEVAL_TOP_K),
    "--qa-top-k", str(QA_TOP_K),
]
qa_seconds = run_cli(qa_args, "qa.log")

## 11. Sonuçlar ve değerlendirme

`retrieval_dataset_test.py` cevapları kaydediyor ama skor hesaplamıyor. Skorları repo'nun kendi değerlendirme modülüyle (`code/src/evaluation/qa_eval.py` → **Exact Match** ve **F1**) hesaplıyorum. Ek olarak iki ölçüt daha veriyorum:

- **İçerme doğruluğu:** altın cevap, normalize edilmiş model cevabının içinde geçiyor mu? (EM, "20 March 851" ↔ "March 20, 851" gibi biçim farklarını cezalandırdığı için.)
- **Destek Recall@k:** sorunun destekleyici paragraflarından kaçı ilk *k* pasajın içinde? (Chunk'lar paragraf sınırlarına göre kesilmediği için eşleştirme, paragrafın ilk 50 alfasayısal karakteri üzerinden yapılıyor.)

In [ ]:
import unicodedata
from src.evaluation.qa_eval import QAExactMatch, QAF1Score
from src.utils.eval_utils import normalize_answer

result = json.load(open(RESULT_PATH, encoding="utf-8"))
records = result["results"]
paragraphs = json.load(open(PARAS_PATH, encoding="utf-8"))
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

def squash(text):
    text = unicodedata.normalize("NFKD", text)
    return re.sub(r"[^a-z0-9]", "", "".join(c for c in text if not unicodedata.combining(c)).lower())

def support_recall(evidence_titles, docs, k):
    joined = [squash(d) for d in docs[:k]]
    keys = [squash(paragraphs[t])[:50] for t in evidence_titles if t in paragraphs]
    return sum(any(key in d for d in joined) for key in keys) / max(1, len(keys))

def evaluate(golds, preds, evidences, doc_lists, ks=(1, 2, 3, 5)):
    em, em_each = QAExactMatch().calculate_metric_scores([[g] for g in golds], preds)
    f1, f1_each = QAF1Score().calculate_metric_scores([[g] for g in golds], preds)
    contain = [float(normalize_answer(g) in normalize_answer(p)) for g, p in zip(golds, preds)]
    out = {"ExactMatch": float(em["ExactMatch"]), "F1": float(f1["F1"]), "İçerme doğruluğu": float(np.mean(contain))}
    for k in ks:
        out[f"Destek Recall@{k}"] = float(np.mean([support_recall(e, d, k) for e, d in zip(evidences, doc_lists)]))
    return out, [x["ExactMatch"] for x in em_each], [x["F1"] for x in f1_each], contain

golds = [r["answer"] for r in records]
evidences = [r["evidence"] for r in records]
mem_scores, em_each, f1_each, contain_each = evaluate(
    golds, [r["predicted_answer"] for r in records], evidences, [r["retrieved_docs"] for r in records])

display(pd.DataFrame([{"tür": r["question_type"], "soru": r["question"], "altın cevap": r["answer"],
                       "MemGraphRAG cevabı": r["predicted_answer"], "EM": em_each[i], "F1": round(f1_each[i], 2),
                       "içerme": contain_each[i],
                       f"destek R@{QA_TOP_K}": round(support_recall(r["evidence"], r["retrieved_docs"], QA_TOP_K), 2)}
                      for i, r in enumerate(records)]))
print("\nMemGraphRAG ortalama skorlar:", {k: round(v, 3) for k, v in mem_scores.items()})

### Tek bir sorunun uçtan uca izi

Aşağıda bir soru için getirilen ilk pasajlar ve modelin ham cevabı (düşünce zinciri + `Answer:`) görülüyor.

In [ ]:
ex = records[0]
print("SORU       :", ex["question"])
print("ALTIN CEVAP:", ex["answer"], "| destekleyici paragraflar:", ex["evidence"], "\n")
for rank, (doc, score) in enumerate(zip(ex["retrieved_docs"][:QA_TOP_K], ex["doc_scores"][:QA_TOP_K]), 1):
    print(f"[{rank}] PPR skoru={score:.4f}\n" + textwrap.fill(doc[:480] + " ...", 120) + "\n")
print("MODELİN HAM CEVABI:\n" + ex["raw_response"])

## 12. Klasik RAG ile küçük bir karşılaştırma

Teknik değerlendirmeyi somutlaştırmak için aynı chunk'lar, aynı embedding modeli, aynı LLM ve **aynı QA istemi** ile bir *klasik (naive) RAG* taban çizgisi kurdum: soru embedding'ine en yakın `QA_TOP_K` chunk doğrudan LLM'e verilir — graf, olgu eşleme ve PPR yoktur. Böylece iki sistem arasındaki tek fark **retrieval yöntemi** olur.

> Not: 10 soru ve ~20 chunk'lık bir derlem istatistiksel bir sonuç çıkarmak için çok küçüktür; bu tablo bir kıyaslama (benchmark) değil, iki yaklaşımın aynı girdide nasıl davrandığını gösteren bir örnektir.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from src.prompts import PromptTemplateManager

work_dir = os.path.join(SAVE_DIR, f"{LLM_NAME.replace('/', '_')}_{EMBEDDING_MODEL.replace('/', '_')}")
chunks = pd.read_parquet(os.path.join(work_dir, "chunk_embeddings", "vdb_chunk.parquet"))
P = np.vstack(chunks["embedding"].values)                      # indekslemede hesaplanan chunk embedding'leri
questions = [r["question"] for r in records]
Q = embedder.batch_encode(questions, norm=True)
dense_docs = [[chunks["content"].iloc[j] for j in np.argsort(-(P @ q))[:RETRIEVAL_TOP_K]] for q in Q]

ptm = PromptTemplateManager(role_mapping={"system": "system", "user": "user", "assistant": "assistant"})
def naive_rag_answer(i):
    prompt_user = "".join(f"Wikipedia Title: {d}\n\n" for d in dense_docs[i][:QA_TOP_K])
    prompt_user += "Question: " + questions[i] + "\nThought: "
    messages = ptm.render(name="rag_qa_musique", prompt_user=prompt_user)      # repo'nun QA istemi
    text = client.chat.completions.create(model=LLM_NAME, messages=messages, temperature=0,
                                          max_tokens=1000).choices[0].message.content
    return text.split("Answer:")[1].strip() if "Answer:" in text else text

with ThreadPoolExecutor(max_workers=4) as ex_:
    dense_preds = list(ex_.map(naive_rag_answer, range(len(questions))))

dense_scores, d_em, d_f1, d_contain = evaluate(golds, dense_preds, evidences, dense_docs)
comparison = pd.DataFrame({"Klasik RAG (yalnız dense retrieval)": dense_scores, "MemGraphRAG": mem_scores}).round(3)
display(comparison)
display(pd.DataFrame([{"soru": q, "altın": g, "Klasik RAG": d, "MemGraphRAG": r["predicted_answer"]}
                      for q, g, d, r in zip(questions, golds, dense_preds, records)]))

In [ ]:
print(f"Toplam indeksleme süresi : {index_seconds:.0f} sn")
print(f"Toplam retrieval+QA süresi: {qa_seconds:.0f} sn")
print("Nihai bellek             :", memory["stats"])
print("Graf                     :", graph["stats"])

## 13. Kullanılan dil modeli ve tercih gerekçesi

**Kullanılan dil modeli: Qwen2.5-7B-Instruct** — Ollama etiketi `qwen2.5:7b` (4-bit `Q4_K_M` GGUF, ~4,7 GB), Colab T4 GPU'sunda yerel olarak çalıştırıldı. Model, sistemdeki bütün LLM rollerinde (NER, üçlü çıkarımı, şema çıkarımı, çelişki tespiti/çözümü ve cevap üretimi) tek başına kullanıldı.

Neden bu model?

| Kriter | Değerlendirme |
|---|---|
| **Colab'da çalıştırılabilirlik** | 4-bit ağırlıklar ~4,7 GB; 8K bağlam ve 4 paralel istekle birlikte T4'ün 15 GB belleğine, `bge-large` embedding modeliyle aynı anda rahatça sığıyor. |
| **MemGraphRAG ile uyum** | Repo yalnızca OpenAI-uyumlu bir `chat/completions` uç noktası bekliyor. Ollama bunu `/v1` altında sağladığı için `--llm-base-url` vermek yetti; repo kodunda modele özel hiçbir değişiklik gerekmedi. |
| **GPU / RAM gereksinimi** | Llama 3.3 70B veya Qwen 32B/72B gibi modeller T4'e sığmıyor; 7B sınıfı, ücretsiz Colab'ın kaldırabildiği en güçlü pratik seçenek. |
| **Cevap kalitesi** | MemGraphRAG'ın indeksleme istemlerinin tamamı **katı JSON** çıktısı istiyor. Qwen2.5 ailesi yapılandırılmış (JSON) çıktı üretimi ve talimat takibi için eğitilmiş, bu boyut sınıfının güçlü modellerinden; daha küçük modellerde bozuk JSON ve eksik üçlü riski artıyor. |
| **Maliyet / API** | Tamamen ücretsiz; API anahtarı, kota veya hız sınırı yok. Notebook'u değerlendiren kişi kendi anahtarını girmeden *Tümünü çalıştır* ile aynı sonucu üretebiliyor. |
| **Açık kaynak** | Apache-2.0 lisanslı açık ağırlıklar; veri Colab oturumunun dışına çıkmıyor. |

Kabul edilen ödünleşimler: 7B'lik nicemlenmiş (quantized) bir model, makalede kullanılan `gpt-4o-mini`'ye göre daha az ve daha gürültülü üçlü çıkarıyor, çelişki tespitinde daha temkinli/tutarsız olabiliyor ve tek GPU'da belirgin biçimde daha yavaş. Bu yüzden derlemi küçük tuttum; makaledeki sayıların yeniden üretilmesi hedeflenmedi. Aynı notebook, 2. bölümdeki iki satır değiştirilerek `gpt-4o-mini` gibi bir API modeliyle de çalıştırılabilir.

## 14. Karşılaşılan hatalar ve çözümleri

Repo'yu çalıştırırken karşılaşılan sorunlar ve uygulanan çözümler:

| # | Hata / sorun | Neden | Çözüm |
|---|---|---|---|
| 1 | `ModuleNotFoundError: No module named 'gritlm'` — `python code/index.py` daha açılışta düşüyor. | BGE kullanılsa bile `code/src/embedding_model/__init__.py` bütün embedding sınıflarını koşulsuz import ediyor; `GritLM.py` de en üstte `from gritlm import GritLM` yapıyor. | `pip install --no-deps gritlm==1.0.2`. `--no-deps` olmadan paket `wandb`, `mteb`, `datasets` gibi gereksiz ve ağır bağımlılıkları çekiyor. |
| 2 | `requirements.txt` Colab ile uyumsuz (`torch==2.5.1`, `transformers==4.45.2`, `vllm==0.6.6.post1`). | Dosya yazarların kendi sunucu ortamına göre sabitlenmiş; `vllm` Colab'daki hazır `torch`'u eski sürüme çekiyor, kurulum uzuyor ve çalışma zamanının yeniden başlatılması gerekiyor. | Dosya olduğu gibi kurulmadı. Kod taranarak gerçekten import edilen paketler çıkarıldı ve yalnızca Colab'da eksik olanlar kuruldu (4. bölüm). `vllm` yalnızca offline OpenIE modu için gerekli; `litellm` ve `boto3` hiç kullanılmıyor. |
| 3 | README'deki komutlar olduğu gibi çalışmıyor: `datasets/corpus/corpus.txt` ve `small-questions.json` yok. | Repo'daki klasörün adı `dataset/` (tekil) ve README'de anılan örnek dosyalar repoda bulunmuyor. | Gerçek `dataset/` klasörü listelendi; indeksleme ve soru dosyaları repo'nun 2Wiki verisinden üretildi (7. bölüm). |
| 4 | `bash code/run_index.sh` → `PYTHON: unbound variable`. | Betik `set -u` ile çalışıyor ama `$PYTHON` değişkenini tanımlamıyor; ayrıca yazarların makinesine ait `/home/wcj/...` yollarını içeriyor. | Kabuk betikleri kullanılmadı; `index.py` ve `retrieval_dataset_test.py` doğrudan çağrıldı (README de bu betikleri "düzenlenebilir şablon" olarak tanımlıyor). |
| 5 | Repo'nun veri setleri `retrieval_dataset_test.py` ile doğrudan yüklenmiyor. | Betik, soruları `[{"source":…, "questions": {tür: [...]}}]` biçiminde bekliyor; `2wikimultihopqa.json` ve `hotpotqa.json` ise ham biçimde (düz liste, `_id`/`context`/`supporting_facts`). | Sorular README'nin "Dataset Format" bölümündeki biçime dönüştürüldü (7. bölüm). |
| 6 | `RuntimeError: index_with_memory failed during OpenIE: division by zero`. | LLM çağrıları başarısız olduğunda (ya da hiç varlık çıkmadığında) `ner()` hatayı yalnızca uyarı olarak geçiyor; ardından `save_openie_results` ortalama varlık uzunluğunu sıfıra bölüyor. Hata iletisi gerçek nedeni gizliyor. | Kök neden LLM tarafındaydı; indekslemeden önce uç noktayı tek bir çağrıyla doğrulayan test hücresi eklendi (5. bölüm). Bu hata görülürse önce LLM uç noktası ve model adı kontrol edilmeli. |
| 7 | Bellek hattı tek bir bozuk JSON cevabında tümüyle duruyor. | `_memory_json` ayrıştıramadığı cevapta istisna fırlatıyor; istisna iş parçacığı havuzundan yukarı taşınıp bütün indekslemeyi sonlandırıyor. Küçük modellerde olasılığı daha yüksek. | `_memory_json` sarmalandı: ayrıştırılamayan cevap uyarı verilip atlanıyor (4. bölümdeki yama). Kaç kez devreye girdiği 8. bölümde raporlanıyor. |
| 8 | Embedding modeli için `AssertionError: Unknown embedding model name`. | Sınıf seçimi model adındaki alt dizgeye göre yapılıyor; ad `bge-large-en-v1.5`, `NV-Embed-v2`, `GritLM` veya `contriever` içermek zorunda. | Model adı tam olarak `BAAI/bge-large-en-v1.5` verildi. Farklı bir BGE sürümü (ör. `bge-m3`) kod değiştirilmeden kullanılamıyor. |

COLAB_EK

**Dikkat edilmesi gereken diğer noktalar** (hata değil ama sonucu etkiliyor):

- `--llm-name` ve `--embedding-model` değerleri indeksleme ve retrieval'da **birebir aynı** olmalı; çıktı klasörünün adı bu ikisinden türetiliyor (`<llm>_<embedding>`), aksi halde `Existing graph was not found` hatası alınıyor.
- LLM cevapları `outputs/<veri>/llm_cache/` altında SQLite'ta önbellekleniyor. Yarıda kesilen bir indeksleme yeniden başlatıldığında tamamlanan çağrılar tekrar yapılmıyor; ancak bozuk bir cevap da önbellekte kaldığı için aynı hatayı yineliyor (madde 7'deki yamanın bir nedeni de bu).
- Ollama'nın varsayılan bağlam penceresi QA istemi için dar; `OLLAMA_CONTEXT_LENGTH` büyütülmezse istem **hata vermeden** kırpılıyor ve cevap kalitesi sessizce düşüyor.

## 15. Teknik değerlendirme: MemGraphRAG, klasik RAG ve GraphRAG'dan nasıl ayrışıyor?

### Üç yaklaşımın karşılaştırması

| | Klasik RAG | GraphRAG (Microsoft GraphRAG / HippoRAG tarzı) | MemGraphRAG |
|---|---|---|---|
| **İndeks birimi** | Birbirinden bağımsız chunk embedding'leri | Chunk'lardan çıkarılan varlık–ilişki grafı (+ topluluk özetleri veya PPR) | **Üç katmanlı bellek** (şema ↔ olgu ↔ pasaj) ve ondan *türetilen* tip–varlık–pasaj grafı |
| **Çıkarım bağlamı** | Yok (LLM indekslemede kullanılmaz) | Her chunk **yalıtık** işlenir; çıktı doğrudan grafa yazılır | Çıkarım çıktısı önce ortak belleğe girer; grafa yazılmadan önce **bütüne bakılarak** soyutlanır, süzülür, çelişkileri giderilir |
| **Soyutlama** | Yok | Topluluk özetleri (MS GraphRAG) ya da yok (HippoRAG) | **Şema katmanı:** her olgu `(tip, ilişki, tip)` düzeyine soyutlanır; tipler grafta ayrı düğümlerdir |
| **Gürültü kontrolü** | Yok | Sınırlı (eş anlamlı varlık birleştirme) | Düşük frekanslı şemalar ve onlara bağlı olgular atılır |
| **Çelişkiler** | Çelişen chunk'lar birlikte getirilir, yük LLM'e kalır | Çelişen kenarlar grafta birlikte yaşar | Pasaj kanıtıyla **tespit edilir** (mutual / temporal / granularity) ve **çözülür** (kept / modified / discarded) |
| **Retrieval** | Soru ↔ chunk benzerliği, top-k | Varlık eşleme + PPR, ya da topluluk özetleri üzerinde map-reduce | Soru ↔ **olgu** benzerliği → tohum varlıklar → tip/varlık/pasaj grafında **PPR** → özgün pasajlar |
| **Çok-adımlı sorular** | Köprü pasaj soruya benzemiyorsa kaçırılır | Graf kenarları üzerinden ulaşılabilir | Graf kenarları + tip düğümleri üzerinden ulaşılabilir; temizlenmiş graf yanlış kenarlara daha az olasılık dağıtır |
| **İndeksleme maliyeti** | Çok düşük (yalnız embedding) | Yüksek (chunk başına LLM; MS GraphRAG'da ayrıca özetleme) | Yüksek: chunk başına 2 çağrı + **olgu başına 1** şema çağrısı + çelişki çağrıları |
| **Sorgu maliyeti** | 1 LLM çağrısı | HippoRAG: 1–2 çağrı; MS GraphRAG global: çok sayıda çağrı | 1 LLM çağrısı (olgu süzme eşikle yapıldığında retrieval LLM'siz) |

### MemGraphRAG'ı ayıran dört nokta

**1. Graf, çıkarımın doğrudan çıktısı değil; belleğin bir türevi.** Klasik GraphRAG hatlarında `chunk → üçlü → graf` akışı tek yönlüdür ve bir chunk'tan çıkan hatalı üçlü olduğu gibi grafa girer. MemGraphRAG'da üçlüler önce olgu katmanına yazılır, her olgu hangi pasajlardan geldiğini ve hangi şemaya bağlı olduğunu taşır. Graf ancak şema çıkarımı → ontoloji filtresi → çelişki çözümü adımlarından *sonra* bu bellekten kurulur (`build_memory_graph`). Makaledeki "çok-etmenli (multi-agent) + paylaşımlı bellek" fikri kodda bu şekilde karşılık buluyor: çıkarıcı, şema çıkarıcı, çelişki dedektörü ve çelişki çözücü ayrı istemlerle çalışan, aynı belleği okuyup yazan LLM rolleri.

**2. Şema katmanı küresel bir bağlam sağlıyor.** Tek tek olgular yereldir; `(<PERSON>, director of, <WORK_OF_ART>)` gibi bir şema ise derlemin tamamında tekrar eden bir örüntüdür. Bu soyutlama iki işe yarıyor: (a) derlemde yalnızca bir-iki kez görülen şemalar büyük olasılıkla çıkarım gürültüsüdür ve atılabilir; (b) tip düğümleri grafta aynı türden varlıkları birbirine bağlayan ek yollar açar. Klasik RAG'da böyle bir küresel görünüm hiç yoktur; MS GraphRAG bunu topluluk özetleriyle, yani çok daha pahalı bir yolla sağlar.

**3. Çelişkiler indeksleme sırasında ele alınıyor.** Klasik RAG ve standart GraphRAG çelişen bilgiyi olduğu gibi saklar; hangisinin doğru olduğuna cevap anında LLM karar vermek zorunda kalır (çoğu zaman da fark etmez). MemGraphRAG aynı özne–ilişkiye sahip farklı nesneli olguları aday olarak eşler, kaynak pasajları kanıt olarak LLM'e gösterir ve gerçek çelişkileri bağlı bileşenler hâlinde çözer. Zamansal çelişkilerde olguyu silmek yerine ilişkiye zaman bilgisi ekleyerek (*modified*) ikisini de koruyabilmesi önemli bir ayrıntı.

**4. Retrieval olgu düzeyinde başlayıp pasaj düzeyinde bitiyor.** Soru önce *olgularla* eşleştirilir (chunk'lardan çok daha ince taneli bir birim), eşleşen olguların varlıkları PPR'ın tohumu olur ve skor graf üzerinden pasajlara yayılır. LLM'e verilen bağlam ise özet değil **özgün pasajlardır**; bu, MS GraphRAG'ın özet tabanlı cevaplarına göre kaynağa daha sadık bir bağlam demektir. Hiçbir olgu eşiği geçemezse sistem kendiliğinden dense retrieval'a düşer; yani en kötü durumda klasik RAG gibi davranır.

### Bu çalışmadaki gözlemler

GOZLEMLER

### Sınırlılıklar ve eleştirel notlar

- **İndeksleme pahalı.** LLM çağrı sayısı olgu sayısıyla doğrusal büyüyor. Bu, yerel bir 7B modelle ancak küçük bir derlemi uygulanabilir kılıyor; büyük derlemler için hızlı bir API modeli veya vLLM ile toplu (batch) çıkarım gerekir. Klasik RAG'ın aynı derlemi saniyeler içinde indekslediği düşünülürse bu ciddi bir ödünleşim.
- **Kalite, çıkarım modeline sıkı sıkıya bağlı.** Şema ve çelişki adımları katı JSON bekliyor; repo'nun özgün kodunda tek bir bozuk cevap bütün indekslemeyi durduruyor (bu notebook'ta yamalandı).
- **Yüzdelik tabanlı ontoloji filtresi küçük derlemlerde kaba kalıyor.** Varsayılan ayar en seyrek %20 şemayı koşulsuz atıyor. Küçük bir derlemde şemaların çoğu zaten bir kez görüldüğü için, atılanlar "gürültü" değil rastgele seçilmiş doğru olgular olabiliyor. Filtre büyük derlemlerde anlamlı; küçüklerde `--ontology-filter-mode absolute` daha güvenli.
- **Çelişki adayları sözcüksel eşleşmeyle bulunuyor** (normalize edilmiş özne + ilişki aynı olmalı). "born in" / "was born in" gibi farklı yazılmış ilişkiler arasındaki çelişkiler aday bile olamıyor.
- **Parçalama belge sınırlarını gözetmiyor.** `index.py` tüm derlemi tek metin olarak sabit 256 token'lık pencerelere bölüyor; bir chunk bir paragrafın sonu ile ilgisiz bir sonrakinin başını birlikte içerebiliyor. Ayrıca chunk'lar embedding modelinin (uncased BERT) tokenizer'ıyla çözüldüğü için metin küçük harfe dönüyor ve noktalama boşlukları değişiyor; LLM özgün metni değil bu hâlini görüyor.
- **Kod tabanı HippoRAG'ın üzerine kurulu;** OpenIE, PPR ve QA kısımları oradan geliyor. MemGraphRAG'ın özgün katkısı bellek hattı ve tip düğümleri; dolayısıyla en adil kıyas HippoRAG ile yapılır ve fark, derlem büyüdükçe (daha fazla tekrar eden şema ve gerçek çelişki oldukça) belirginleşir.

**Özetle:** Klasik RAG "benzer metni bul", GraphRAG "metinden graf çıkar ve grafta gez" derken, MemGraphRAG araya bir **bellek ve denetim katmanı** koyuyor: çıkarılan bilgiyi önce soyutluyor, sıklığına göre süzüyor, kendi içinde tutarlı hâle getiriyor, grafı ancak ondan sonra kuruyor. Kazancı daha temiz ve tutarlı bir graf ile çok-adımlı sorularda daha isabetli retrieval; bedeli ise belirgin biçimde daha pahalı ve çıkarım modelinin kalitesine daha duyarlı bir indeksleme.

### Kaynaklar

- Wu, C., Xiang, Z., Tang, Y., Chen, Z., Zhang, Q., Su, J. *MemGraphRAG: Memory-based Multi-Agent System for Graph Retrieval-Augmented Generation.* KDD 2026. [arXiv:2606.00610](https://arxiv.org/abs/2606.00610)
- Kaynak kod: [github.com/XMUDeepLIT/MemGraphRAG](https://github.com/XMUDeepLIT/MemGraphRAG)
- Üzerine kurulduğu çalışma: [HippoRAG](https://github.com/OSU-NLP-Group/HippoRAG)
- Dil modeli: [Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) · Sunucu: [Ollama](https://ollama.com/library/qwen2.5) · Embedding: [BAAI/bge-large-en-v1.5](https://huggingface.co/BAAI/bge-large-en-v1.5)